# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
print(f"Dataset title: {dataset.metadata.name}\n{dataset.metadata.description}")
print(f"Published: {dataset.metadata.datePublished}\nVersion: {dataset.metadata.version}")
print(f"DOI/Identifier: {dataset.metadata.identifier}")


## 2. Data Overview
Review available record sets, fields, and their `@id`s (identifiers).

In [ ]:
# List all available record sets
record_sets = dataset.metadata.recordSets

print("Record Sets in the dataset:")
for rset in record_sets:
    print(f"  - @id: {rset.id} | name: {rset.name}")

# Pick the main table (first record set)
main_record_set = record_sets[0]
print(f"\nMain record set: {main_record_set.name} (id={main_record_set.id})")

# List all fields in the main record set
print("Fields and their @id in this record set:")
for field in main_record_set.fields:
    field_type = getattr(field, 'dataType', None)
    print(f"  - @id: {field.id} | name: {field.name} | dataType: {field_type}")

## 3. Data Extraction
Load data from the main record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

In [ ]:
# For flexibility, extract all record sets into dataframes using their @id
dataframes = {}

for recset in record_sets:
    # Use the @id of the record set
    record_set_id = recset.id
    print(f"Loading records from record set {record_set_id} ...")
    try:
        records = list(dataset.records(record_set=record_set_id))
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Loaded: {len(df)} rows, fields: {df.columns.tolist()}")
    except Exception as e:
        print(f"  Could not load: {e}")

# For demo, pick the main record set @id and show its columns and top rows
main_id = main_record_set.id
print(f"\nColumns in main record set ({main_id}):\n{dataframes[main_id].columns.tolist()}")
dataframes[main_id].head()

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.

### Example: Numeric filtering and normalization

In [ ]:
# Let's choose a numeric field (e.g. 'MW [kDa]' for Molecular Weight) by its @id
# We'll discover it from the previous field listing. Suppose its @id is 'MW (kDa)'.
numeric_field_id = None
for field in main_record_set.fields:
    # Pick a field with dataType Float or Integer and plausible name
    dt = getattr(field, 'dataType', '')
    if dt and dt.lower() in ('float', 'number', 'integer'):
        if 'mw' in field.name.lower() or 'abundance' in field.name.lower():
            numeric_field_id = field.id
            print(f"Selected numeric field @id: {numeric_field_id} (name: {field.name})")
            break

if not numeric_field_id:
    raise Exception("No suitable numeric field found for EDA.")

# Filtering - let's filter with a threshold (e.g., MW > 10 kDa)
threshold = 10
main_df = dataframes[main_id]

# Ensure the column name matches the field's @id
if numeric_field_id not in main_df.columns:
    print(f"Column {numeric_field_id} not in DataFrame columns.")
else:
    filtered_df = main_df[pd.to_numeric(main_df[numeric_field_id], errors='coerce') > threshold].copy()
    print(f"Filtered records with {numeric_field_id} > {threshold}:")
    print(filtered_df[[numeric_field_id]].head())
    
    # Normalization
    col = pd.to_numeric(filtered_df[numeric_field_id], errors='coerce')
    filtered_df[f"{numeric_field_id}_normalized"] = (col - col.mean()) / col.std()
    print(f"Normalized {numeric_field_id}:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())
    
    # If there is a field to group by (e.g., protein description or type)
    # We'll search for a possible categorical field
    group_field_id = None
    for field in main_record_set.fields:
        if getattr(field, 'dataType', '') == 'Text' and 'desc' in field.name.lower():
            group_field_id = field.id
            break
    if group_field_id and group_field_id in filtered_df.columns:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Mean {numeric_field_id} grouped by {group_field_id}:")
        print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot distribution of the numeric field
if numeric_field_id in main_df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(pd.to_numeric(main_df[numeric_field_id], errors='coerce').dropna(), bins=30, kde=True)
    plt.title(f'Distribution of {numeric_field_id}')
    plt.xlabel(numeric_field_id)
    plt.ylabel('Count')
    plt.show()

# If normalized column exists, plot normalized as well
if f"{numeric_field_id}_normalized" in filtered_df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(filtered_df[f"{numeric_field_id}_normalized"].dropna(), bins=30, kde=True, color='orange')
    plt.title(f'Normalized {numeric_field_id} (> {threshold})')
    plt.xlabel(f"{numeric_field_id} (normalized)")
    plt.show()

## 6. Conclusion
This notebook demonstrated how to load, inspect, filter, and visualize a FAIR-compliant proteomics dataset using the `mlcroissant` library. We:
- Explored the available record sets and fields by their `@id`
- Loaded the main data table to a pandas DataFrame
- Selected a numeric field using its `@id` and performed filtering and normalization
- Visualized the distributions

Further analyses can include joining with other record sets, advanced filtering, or integrating external biological knowledge. For production ML, always refer to the original Croissant schema for field definitions and data provenance.
